In [ ]:
#@title ⚡ Khởi Động OmniVoice GPU Worker & Đường Hầm Tunnel (Bấm Run)
#@markdown Chọn phương thức kết nối về máy local của bạn:
#@markdown - **ngrok**: Khuyên dùng! Dùng **Static Domain cố định**, cấu hình file `.env` 1 lần dùng mãi mãi.
#@markdown - **cloudflare**: Miễn phí không cần tài khoản, link ngẫu nhiên mỗi lần chạy.
TUNNEL_METHOD = "ngrok" #@param ["ngrok", "cloudflare"]
NGROK_AUTHTOKEN = "3JJU5ZC5zThuSkjjTF4SMZRum1r_5jStVjtjmpZG5CXdMsCih" #@param {type:"string"}
NGROK_STATIC_DOMAIN = "tipper-semantic-dropper.ngrok-free.dev" #@param {type:"string"}

import os
import sys
import time
import re
import subprocess
import urllib.request
import json
from IPython.display import display, HTML

# 1. Kiểm tra GPU CUDA
print("🔍 [1/5] Đang kiểm tra card đồ hoạ GPU...")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("❌ Chưa bật GPU! Vui lòng vào menu: Runtime -> Change runtime type -> Chọn T4 GPU rồi bấm chạy lại ô này.")
print("✅ Nhận diện GPU thành công!\n" + gpu_check.stdout.split("\n")[8])

# 2. Cài đặt các thư viện cần thiết
print("\n📦 [2/5] Đang cài đặt thư viện cần thiết (OmniVoice, Accelerate, Transformers, FastAPI, Uvicorn, PyNgrok)...")
reqs = [
    "omnivoice", "accelerate>=0.34.0", "webdataset", "safetensors", "transformers>=4.40.0",
    "soundfile", "librosa", "pydub", "scipy", "av>=11.0.0", "faster-whisper>=1.0.0", "fastapi", "uvicorn[standard]",
    "python-multipart", "httpx", "pyngrok"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("✅ Đã cài đặt xong toàn bộ thư viện phụ thuộc!")

# 3. Thiết lập mã nguồn OmniVoice Colab Worker mới nhất
print("\n⚙️ [3/5] Đang đồng bộ mã nguồn OmniVoice Colab Worker mới nhất...")
worker_url = "https://raw.githubusercontent.com/tranvankha1989/TranslateVideo/main/backend/colab_worker.py"
loaded_from_github = False
try:
    urllib.request.urlretrieve(worker_url, "colab_worker.py")
    if os.path.exists("colab_worker.py") and os.path.getsize("colab_worker.py") > 1000:
        loaded_from_github = True
        print("✅ Đã đồng bộ thành công colab_worker.py mới nhất từ GitHub!")
except Exception as e:
    print(f"⚠️ Không thể tải trực tiếp từ GitHub ({e}), sử dụng mã nguồn tích hợp sẵn...")

if not loaded_from_github:
    print("📝 Đang tạo colab_worker.py từ mã nguồn tích hợp...")
    worker_code = '"""\ncolab_worker.py — OmniVoice GPU Remote Worker Server\n────────────────────────────────────────────────────\nFile này được thiết kế để chạy trên Google Colab (với GPU NVIDIA Tesla T4 16GB)\nhoặc bất kỳ máy chủ có GPU nào.\n\nCung cấp các API:\n  - GET  /api/remote/health    : Kiểm tra trạng thái máy chủ & GPU VRAM\n  - POST /api/remote/prompt    : Trích xuất đặc trưng giọng nói (VoiceClonePrompt .pt)\n  - POST /api/remote/generate  : Sinh âm thanh đa chunk siêu tốc bằng OmniVoice trên GPU\n\nKhởi chạy độc lập:\n  python colab_worker.py --port 8000\n"""\n\nimport os\nimport io\nimport gc\nimport re\nimport sys\nimport glob\nimport ctypes\nimport tempfile\nimport logging\nfrom pathlib import Path\n\n# pyrefly: ignore [missing-import]\nimport torch\n# pyrefly: ignore [missing-import]\nimport soundfile as sf\n# pyrefly: ignore [missing-import]\nimport numpy as np\nfrom fastapi import FastAPI, HTTPException, UploadFile, File, Form\nfrom fastapi.responses import Response, JSONResponse\nfrom fastapi.middleware.cors import CORSMiddleware\n\nfrom omnivoice import OmniVoice, VoiceClonePrompt\n\nlogging.basicConfig(\n    level=logging.INFO,\n    format="%(asctime)s [%(levelname)s] colab_worker — %(message)s",\n    datefmt="%H:%M:%S",\n)\nlogger = logging.getLogger("colab_worker")\n\nSAMPLE_RATE = 24_000\nMODEL_ID = os.getenv("OMNIVOICE_MODEL_ID", "k2-fsa/OmniVoice")\n\ndef preload_cuda_libraries():\n    """Tự động tìm nạp các file libcublas.so, libcudnn.so từ site-packages vào bộ nhớ tiến trình trên Linux/Colab."""\n    for p in sys.path:\n        for lib in glob.glob(os.path.join(p, "nvidia", "*", "lib", "*.so*")):\n            try:\n                ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)\n            except Exception:\n                pass\n\npreload_cuda_libraries()\n\napp = FastAPI(\n    title="OmniVoice Colab GPU Worker",\n    description="GPU Inference Worker phục vụ ứng dụng self-tts qua Ngrok / Cloudflare Tunnel",\n    version="1.2.0",\n)\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=True,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n_model: OmniVoice | None = None\n_whisper_model = None\n_whisper_model_size: str | None = None\n\n\ndef get_gpu_info() -> dict:\n    """Lấy thông tin card đồ hoạ và VRAM khả dụng."""\n    if not torch.cuda.is_available():\n        return {\n            "device": "cpu",\n            "gpu_name": "No GPU (CPU Mode)",\n            "vram_total_gb": 0.0,\n            "vram_free_gb": 0.0,\n        }\n    dev_id = 0\n    props = torch.cuda.get_device_properties(dev_id)\n    total_mem = props.total_memory / (1024 ** 3)\n    free_mem, _ = torch.cuda.mem_get_info(dev_id)\n    free_gb = free_mem / (1024 ** 3)\n    return {\n        "device": f"cuda:{dev_id}",\n        "gpu_name": torch.cuda.get_device_name(dev_id),\n        "vram_total_gb": round(total_mem, 2),\n        "vram_free_gb": round(free_gb, 2),\n    }\n\n\ndef clean_vram():\n    """Giải phóng bộ nhớ VRAM và thu dọn rác (Smart VRAM Management)."""\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        torch.cuda.ipc_collect()\n\n\ndef get_whisper(model_size: str = "large-v3-turbo", force_cpu: bool = False):\n    """Nạp Faster-Whisper trên GPU với cơ chế lazy load và dự phòng CPU tự động."""\n    global _whisper_model, _whisper_model_size\n    target = model_size.strip().lower()\n    target_key = f"{target}_cpu" if force_cpu else target\n    if _whisper_model is not None and _whisper_model_size == target_key:\n        return _whisper_model\n\n    from faster_whisper import WhisperModel\n    clean_vram()\n    if force_cpu or not torch.cuda.is_available():\n        logger.info(f"🎙️ [Whisper STT] Đang tải Faster-Whisper \'{target}\' (CPU Mode)...")\n        _whisper_model = WhisperModel(\n            target,\n            device="cpu",\n            compute_type="int8",\n            cpu_threads=4,\n            num_workers=2,\n        )\n    else:\n        logger.info(f"🎙️ [Whisper STT] Đang tải Faster-Whisper \'{target}\' lên CUDA (float16)...")\n        try:\n            _whisper_model = WhisperModel(\n                target,\n                device="cuda",\n                compute_type="float16",\n                cpu_threads=4,\n                num_workers=1,\n            )\n        except Exception as e:\n            logger.warning(f"⚠️ Không thể tải Faster-Whisper trên CUDA ({e}), tự động chuyển sang CPU...")\n            _whisper_model = WhisperModel(\n                target,\n                device="cpu",\n                compute_type="int8",\n                cpu_threads=4,\n                num_workers=2,\n            )\n    _whisper_model_size = target_key\n    logger.info("✅ Faster-Whisper đã sẵn sàng phục vụ!")\n    return _whisper_model\n\n\ndef load_worker_model() -> None:\n    """Tải mô hình OmniVoice vào GPU VRAM."""\n    global _model\n    if _model is not None:\n        return\n\n    device = "cuda:0" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if torch.cuda.is_available() else torch.float32\n\n    if torch.cuda.is_available():\n        torch.backends.cudnn.benchmark = True\n\n    gpu_info = get_gpu_info()\n    logger.info(\n        f"🚀 Đang tải mô hình {MODEL_ID} lên {device} ({gpu_info.get(\'gpu_name\')}) [dtype={dtype}]..."\n    )\n\n    clean_vram()\n    _model = OmniVoice.from_pretrained(\n        MODEL_ID,\n        device_map=device,\n        dtype=dtype,\n    )\n    logger.info("✅ OmniVoice đã sẵn sàng phục vụ trên GPU!")\n\n\n@app.on_event("startup")\ndef startup_event():\n    load_worker_model()\n\n\n@app.get("/api/remote/health")\n@app.get("/gradio_api/remote/health")\ndef health_check():\n    """Endpoint kiểm tra tình trạng kết nối và card GPU."""\n    gpu = get_gpu_info()\n    return {\n        "status": "ok",\n        "model_loaded": _model is not None,\n        "whisper_loaded": _whisper_model is not None,\n        "sample_rate": SAMPLE_RATE,\n        "provider": "Google Colab Worker",\n        **gpu,\n    }\n\n\ndef clean_vietnamese_text(text: str) -> str:\n    """Làm sạch và chuẩn hóa văn bản."""\n    if not text:\n        return ""\n    text = re.sub(r":\\s*", ", ", text)\n    text = re.sub(r";\\s*", ", ", text)\n    for q in [\'"\', \'“\', \'”\', "\'", \'‘\', \'’\', \'«\', \'»\']:\n        text = text.replace(q, "")\n    text = re.sub(r"\\.{2,}", ".", text)\n    text = re.sub(r"-{2,}", "-", text)\n    text = re.sub(r"[ \\t]+", " ", text).strip()\n    return text\n\n\ndef split_into_chunks(text: str, max_chars: int = 450) -> list[str]:\n    """Chia nhỏ văn bản thành các đoạn tự nhiên theo câu."""\n    cleaned = clean_vietnamese_text(text)\n    paragraphs = [p.strip() for p in re.split(r"\\n+", cleaned) if p.strip()]\n\n    chunks: list[str] = []\n    for p in paragraphs:\n        if len(p) <= max_chars:\n            chunks.append(p)\n            continue\n\n        sentences = re.split(r"(?<=[.?!…])\\s+", p)\n        cur = ""\n        for s in sentences:\n            s = s.strip()\n            if not s:\n                continue\n            if not cur:\n                cur = s\n            elif len(cur) + len(s) + 1 <= max_chars:\n                cur += " " + s\n            else:\n                chunks.append(cur)\n                cur = s\n        if cur:\n            chunks.append(cur)\n\n    final_chunks: list[str] = []\n    for c in chunks:\n        c = c.strip()\n        if c and not c.endswith((".", "!", "?", "…")):\n            c += "."\n        if c:\n            final_chunks.append(c)\n\n    return final_chunks or [cleaned]\n\n\n@app.post("/api/remote/prompt")\n@app.post("/gradio_api/remote/prompt")\nasync def create_prompt_endpoint(\n    audio_file: UploadFile = File(...),\n    ref_text: str | None = Form(default=None),\n):\n    """\n    Trích xuất VoiceClonePrompt (.pt) từ file audio tham chiếu sử dụng GPU.\n    Trả về nội dung nhị phân (binary) của file prompt .pt.\n    """\n    if _model is None:\n        raise HTTPException(status_code=503, detail="Mô hình OmniVoice chưa được tải")\n\n    suffix = Path(audio_file.filename or "sample.wav").suffix or ".wav"\n    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp_audio:\n        content = await audio_file.read()\n        tmp_audio.write(content)\n        tmp_audio_path = tmp_audio.name\n\n    try:\n        clean_text = ref_text.strip() if (ref_text and ref_text.strip()) else None\n        logger.info(f"🎙️ Đang trích xuất VoiceClonePrompt trên GPU cho file: {audio_file.filename}...")\n        \n        prompt = _model.create_voice_clone_prompt(\n            ref_audio=tmp_audio_path,\n            ref_text=clean_text,\n            preprocess_prompt=True,\n        )\n\n        with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp_pt:\n            prompt.save(tmp_pt.name)\n            with open(tmp_pt.name, "rb") as pf:\n                content = pf.read()\n            os.remove(tmp_pt.name)\n\n        logger.info("✅ Trích xuất VoiceClonePrompt thành công!")\n        return Response(content=content, media_type="application/octet-stream")\n    finally:\n        try:\n            os.remove(tmp_audio_path)\n        except OSError:\n            pass\n        clean_vram()\n\n\n@app.post("/api/remote/transcribe")\n@app.post("/gradio_api/remote/transcribe")\nasync def transcribe_endpoint(\n    audio_file: UploadFile = File(...),\n    language: str | None = Form(default=None),\n    model_size: str | None = Form(default="large-v3-turbo"),\n    initial_prompt: str | None = Form(default=None),\n    vad_filter: bool = Form(default=True),\n    vad_threshold: float = Form(default=0.50),\n    min_speech_duration_ms: int = Form(default=150),\n    min_silence_duration_ms: int = Form(default=350),\n    speech_pad_ms: int = Form(default=150),\n    beam_size: int = Form(default=3),\n):\n    """\n    Bóc tách phụ đề và nhận diện giọng nói sử dụng Faster-Whisper trên Colab GPU.\n    Áp dụng Smart VRAM: Nạp mô hình theo nhu cầu, tự động fallback sang CPU nếu CUDA gặp lỗi,\n    và dọn sạch VRAM sau khi hoàn thành.\n    """\n    suffix = Path(audio_file.filename or "audio.wav").suffix or ".wav"\n    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp_audio:\n        content = await audio_file.read()\n        tmp_audio.write(content)\n        tmp_audio_path = tmp_audio.name\n\n    def _do_transcribe(whisper_inst):\n        # Nạp audio đầu vào an toàn: nạp bằng soundfile/numpy float32 16kHz để bỏ qua hoàn toàn lỗi PyAV metadata_errors\n        try:\n            audio_data, sr = sf.read(tmp_audio_path, dtype="float32")\n            if audio_data.ndim > 1:\n                audio_data = np.mean(audio_data, axis=1)\n            if sr != 16000:\n                import librosa\n                audio_data = librosa.resample(audio_data, orig_sr=sr, target_sr=16000)\n            audio_input = audio_data\n        except Exception as load_err:\n            logger.warning(f"⚠️ Tiền xử lý audio bằng soundfile gặp lỗi ({load_err}), sử dụng đường dẫn file trực tiếp: {tmp_audio_path}")\n            audio_input = tmp_audio_path\n\n        segments_gen, info = whisper_inst.transcribe(\n            audio_input,\n            language=lang_arg,\n            initial_prompt=chinese_prompt,\n            beam_size=beam_size,\n            best_of=beam_size,\n            condition_on_previous_text=False,\n            repetition_penalty=1.2,\n            no_speech_threshold=0.85,\n            log_prob_threshold=-1.5,\n            compression_ratio_threshold=2.8,\n            vad_filter=vad_filter,\n            vad_parameters=dict(\n                threshold=vad_threshold,\n                min_speech_duration_ms=min_speech_duration_ms,\n                min_silence_duration_ms=min_silence_duration_ms,\n                speech_pad_ms=speech_pad_ms,\n            ) if vad_filter else None,\n            word_timestamps=True,\n        )\n\n        result_segments = []\n        for i, s in enumerate(segments_gen):\n            words_data = []\n            if getattr(s, "words", None):\n                for w in s.words:\n                    w_text = getattr(w, "word", "").strip()\n                    if not w_text:\n                        continue\n                    words_data.append({\n                        "word": w_text,\n                        "start": round(float(getattr(w, "start", 0.0)), 3),\n                        "end": round(float(getattr(w, "end", 0.0)), 3),\n                        "probability": round(float(getattr(w, "probability", 1.0)), 2),\n                    })\n\n            txt = getattr(s, "text", "").strip()\n            if not txt and not words_data:\n                continue\n\n            # Mốc thời gian chính xác theo từ (không bị trễ đầu và cắt cụt đuôi)\n            if words_data:\n                for wi in range(len(words_data)):\n                    w_item = words_data[wi]\n                    wt = w_item.get("word", "").strip()\n                    ws = float(w_item.get("start", 0.0))\n                    we = float(w_item.get("end", ws + 0.3))\n                    wdur = max(0.1, we - ws)\n                    is_c = any(\'\\u4e00\' <= ch <= \'\\u9fff\' or \'\\u3040\' <= ch <= \'\\u30ff\' for ch in wt)\n                    wmax = max(0.65, len(wt) * 0.45 + 0.3) if is_c else max(0.85, len(wt) * 0.25 + 0.4)\n                    if wdur > wmax:\n                        if wi + 1 < len(words_data):\n                            nxt_s = float(words_data[wi + 1].get("start", we))\n                            if nxt_s > ws + wmax:\n                                w_item["end"] = round(ws + wmax, 3)\n                        else:\n                            w_item["end"] = round(ws + wmax, 3)\n\n                valid_w = [w for w in words_data if w.get("word", "").strip()]\n                if valid_w:\n                    seg_start = valid_w[0]["start"]\n                    seg_end = valid_w[-1]["end"]\n                else:\n                    seg_start = round(float(getattr(s, "start", 0.0)), 3)\n                    seg_end = round(float(getattr(s, "end", 0.0)), 3)\n            else:\n                seg_start = round(float(getattr(s, "start", 0.0)), 3)\n                seg_end = round(float(getattr(s, "end", 0.0)), 3)\n\n            if seg_end <= seg_start:\n                seg_end = round(seg_start + 0.3, 3)\n\n            result_segments.append({\n                "id": i + 1,\n                "seek": getattr(s, "seek", 0),\n                "start": round(seg_start, 3),\n                "end": round(seg_end, 3),\n                "text": txt,\n                "words": words_data,\n            })\n\n        det_lang = getattr(info, "language", lang_arg or "vi")\n        return result_segments, det_lang, info\n\n    try:\n        lang_arg = None if (not language or language == "auto") else language.split("-")[0]\n        chinese_prompt = initial_prompt.strip() if initial_prompt and initial_prompt.strip() else None\n        if lang_arg == "zh" and not chinese_prompt:\n            chinese_prompt = "以下是普通话的句子，请用简体中文输出。"\n        \n        logger.info(\n            f"🎙️ [Colab Whisper STT] Nhận diện file \'{audio_file.filename}\' (Lang: {lang_arg or \'auto\'}, Model: {model_size or \'large-v3-turbo\'}, VAD Threshold: {vad_threshold})..."\n        )\n\n        try:\n            whisper = get_whisper(model_size or "large-v3-turbo", force_cpu=False)\n            result_segments, det_lang, info = _do_transcribe(whisper)\n        except Exception as e:\n            logger.warning(f"⚠️ Nhận diện bằng CUDA thất bại ({e}), tự động chuyển sang chế độ CPU dự phòng...")\n            whisper_cpu = get_whisper(model_size or "large-v3-turbo", force_cpu=True)\n            result_segments, det_lang, info = _do_transcribe(whisper_cpu)\n\n        logger.info(f"✅ [Colab Whisper STT] Đã nhận diện {len(result_segments)} câu (Ngôn ngữ: {det_lang})!")\n        return JSONResponse(content={\n            "status": "ok",\n            "language": det_lang,\n            "language_probability": round(float(getattr(info, "language_probability", 1.0)), 3),\n            "duration": round(float(getattr(info, "duration", 0.0)), 2),\n            "segments": result_segments,\n        })\n    except Exception as exc:\n        import traceback\n        err_detail = traceback.format_exc()\n        logger.error(f"❌ [Colab Transcribe Error]: {err_detail}")\n        return JSONResponse(\n            status_code=500,\n            content={"status": "error", "message": str(exc), "traceback": err_detail},\n        )\n    finally:\n        try:\n            os.remove(tmp_audio_path)\n        except OSError:\n            pass\n        clean_vram()\n\n\n@app.post("/api/remote/generate")\n@app.post("/gradio_api/remote/generate")\nasync def generate_endpoint(\n    text: str = Form(...),\n    mode: str = Form(default="clone"),\n    instruct: str | None = Form(default=None),\n    num_step: int = Form(default=16),\n    cfg_value: float = Form(default=2.0),\n    speed: float = Form(default=1.0),\n    seed: int | None = Form(default=42),\n    ref_text: str | None = Form(default=None),\n    prompt_file: UploadFile | None = File(default=None),\n    ref_audio_file: UploadFile | None = File(default=None),\n):\n    """\n    Sinh âm thanh đa chunk bằng GPU và trả về dữ liệu âm thanh 24kHz dạng WAV binary.\n    """\n    if _model is None:\n        raise HTTPException(status_code=503, detail="Mô hình OmniVoice chưa được tải")\n\n    if seed is not None:\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n\n    voice_clone_prompt: VoiceClonePrompt | None = None\n    tmp_ref_path: str | None = None\n\n    try:\n        # 1. Nạp VoiceClonePrompt nếu được gửi lên\n        if prompt_file is not None:\n            pt_bytes = await prompt_file.read()\n            with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp_pt:\n                tmp_pt.write(pt_bytes)\n                tmp_pt_path = tmp_pt.name\n            try:\n                device_str = "cuda:0" if torch.cuda.is_available() else "cpu"\n                voice_clone_prompt = VoiceClonePrompt.load(tmp_pt_path, map_location=device_str)\n            except Exception as e:\n                logger.warning(f"Không thể đọc prompt_file bằng VoiceClonePrompt.load: {e}")\n            finally:\n                try:\n                    os.remove(tmp_pt_path)\n                except OSError:\n                    pass\n\n        # 2. Hoặc nạp file ref_audio nếu được gửi lên\n        if ref_audio_file is not None:\n            sfx = Path(ref_audio_file.filename or "ref.wav").suffix or ".wav"\n            with tempfile.NamedTemporaryFile(delete=False, suffix=sfx) as tmp_ref:\n                tmp_ref.write(await ref_audio_file.read())\n                tmp_ref_path = tmp_ref.name\n\n        chunks = split_into_chunks(text, max_chars=450)\n        logger.info(\n            f"⚡ Colab GPU | Mode={mode} | num_step={num_step} | Chunks={len(chunks)} | Text: \'{text[:50]}…\'"\n        )\n\n        all_audios: list[np.ndarray] = []\n        design_voice_clone_prompt: VoiceClonePrompt | None = None\n\n        with torch.inference_mode():\n            for idx, chunk in enumerate(chunks):\n                gen_kwargs = {\n                    "text": chunk,\n                    "language": "vi",\n                    "num_step": num_step,\n                    "guidance_scale": cfg_value,\n                    "normalize_text": False,\n                    "speed": speed,\n                }\n\n                if mode == "clone":\n                    if voice_clone_prompt is not None:\n                        gen_kwargs["voice_clone_prompt"] = voice_clone_prompt\n                    elif tmp_ref_path:\n                        gen_kwargs["ref_audio"] = tmp_ref_path\n                        if ref_text and ref_text.strip():\n                            gen_kwargs["ref_text"] = clean_vietnamese_text(ref_text)\n                elif mode == "design":\n                    if design_voice_clone_prompt is not None:\n                        gen_kwargs["voice_clone_prompt"] = design_voice_clone_prompt\n                    elif instruct:\n                        gen_kwargs["instruct"] = instruct\n\n                audio_list = _model.generate(**gen_kwargs)\n                if audio_list and len(audio_list) > 0:\n                    audio_np = np.array(audio_list[0], dtype=np.float32)\n                    if audio_np.ndim > 1:\n                        audio_np = audio_np.squeeze()\n                    all_audios.append(audio_np)\n\n        if not all_audios:\n            raise HTTPException(status_code=500, detail="Mô hình không sinh được âm thanh")\n\n        # Ghép các đoạn audio lại với khoảng lặng 0.22s giữa các câu\n        silence_samples = int(SAMPLE_RATE * 0.22)\n        silence_array = np.zeros(silence_samples, dtype=np.float32)\n\n        final_pieces: list[np.ndarray] = []\n        for i, a in enumerate(all_audios):\n            # Mờ dần 10ms ở đầu và đuôi câu để khử tiếng click nổ\n            fade_len = int(SAMPLE_RATE * 0.01)\n            if len(a) > fade_len * 2:\n                fade_in = np.linspace(0, 1, fade_len, dtype=np.float32)\n                fade_out = np.linspace(1, 0, fade_len, dtype=np.float32)\n                a[:fade_len] *= fade_in\n                a[-fade_len:] *= fade_out\n\n            final_pieces.append(a)\n            if i < len(all_audios) - 1:\n                final_pieces.append(silence_array)\n\n        final_audio = np.concatenate(final_pieces)\n\n        # Xuất ra bộ nhớ đệm WAV 24kHz\n        wav_buf = io.BytesIO()\n        sf.write(wav_buf, final_audio, SAMPLE_RATE, format="WAV")\n        wav_buf.seek(0)\n\n        dur = round(len(final_audio) / SAMPLE_RATE, 2)\n        logger.info(f"🎉 Hoàn tất sinh {dur}s audio trên GPU T4!")\n\n        return Response(content=wav_buf.getvalue(), media_type="audio/wav")\n\n    finally:\n        if tmp_ref_path:\n            try:\n                os.remove(tmp_ref_path)\n            except OSError:\n                pass\n        clean_vram()\n\n\nif __name__ == "__main__":\n    import argparse\n    import uvicorn\n\n    parser = argparse.ArgumentParser(description="OmniVoice Colab GPU Worker Server")\n    parser.add_argument("--host", default="0.0.0.0", help="Host lắng nghe")\n    parser.add_argument("--port", type=int, default=8000, help="Cổng dịch vụ")\n    args = parser.parse_args()\n\n    uvicorn.run(app, host=args.host, port=args.port)\n'\n    with open("colab_worker.py", "w", encoding="utf-8") as f:
        f.write(worker_code)
    print("✅ Đã ghi mã nguồn colab_worker.py thành công!")

# 4. Khởi động server ngầm và ghi log
print("\n🚀 [4/5] Đang khởi động OmniVoice GPU Worker & nạp mô hình vào VRAM...")
log_file = open("worker.log", "w", encoding="utf-8")
server_proc = subprocess.Popen([sys.executable, "-u", "colab_worker.py"], stdout=log_file, stderr=subprocess.STDOUT)

# Chờ server sẵn sàng bằng cách poll health endpoint (tối đa 90s)
is_ready = False
print("⏳ Đang tải trọng số mô hình k2-fsa/OmniVoice (vui lòng đợi ~30-60 giây)...", end="", flush=True)
for _ in range(45):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/remote/health", timeout=2) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                is_ready = True
                print(f"\n✅ Server đã sẵn sàng! [{data.get('gpu_name')} - {data.get('vram_total_gb')}GB VRAM]")
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)

if not is_ready:
    log_file.close()
    with open("worker.log", "r", encoding="utf-8") as f:
        err_output = f.read()
    raise RuntimeError(f"❌ Server không khởi động được! Chi tiết log lỗi:\n\n{err_output}")

# 5. Khởi chạy Đường hầm Tunnel (Ngrok Static Domain hoặc Cloudflare Tunnel)
tunnel_url = None

if TUNNEL_METHOD == "ngrok" and NGROK_AUTHTOKEN.strip():
    print("\n🌐 [5/5] Đang kết nối đường hầm bảo mật qua Ngrok...")
    try:
        from pyngrok import ngrok, conf
        ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
        domain_val = NGROK_STATIC_DOMAIN.strip() if NGROK_STATIC_DOMAIN.strip() else None
        if domain_val:
            print(f"🔗 Đang kết nối tới tên miền tĩnh Ngrok: {domain_val} ...")
            tunnel = ngrok.connect(8000, "http", domain=domain_val)
        else:
            print("🔗 Đang tạo đường hầm Ngrok ngẫu nhiên...")
            tunnel = ngrok.connect(8000, "http")
        tunnel_url = tunnel.public_url
    except Exception as ne:
        print(f"⚠️ Lỗi kết nối Ngrok ({ne}). Đang tự động chuyển sang Cloudflare Tunnel dự phòng...")

if not tunnel_url:
    print("\n🌐 [5/5] Đang kết nối đường hầm Cloudflare Tunnel dự phòng...")
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
    subprocess.run(["dpkg", "-i", "-E", "cloudflared-linux-amd64.deb"], capture_output=True)
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    start_time = time.time()
    while time.time() - start_time < 30:
        line = tunnel_proc.stdout.readline()
        if not line: continue
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    display(HTML('<h1><font color="green">✅ Kết nối thành công!</font></h1>'))
    is_static = bool(TUNNEL_METHOD == "ngrok" and NGROK_STATIC_DOMAIN.strip() and NGROK_STATIC_DOMAIN.strip() in tunnel_url)
    print("\n" + "="*70)
    print("🎉 GOOGLE COLAB GPU (T4 16GB) ĐÃ SẴN SÀNG HOẠT ĐỘNG!")
    print(f"👉 URL kết nối: {tunnel_url}")
    print("="*70)
    if is_static:
        print("\n✨ BẠN ĐANG DÙNG NGROK STATIC DOMAIN CỐ ĐỊNH!")
        print("💡 Chỉ cần cấu hình file backend/.env trên máy bạn 1 LẦN DUY NHẤT:")
        print("-"*70)
        print("USE_COLAB_GPU=true")
        print(f"COLAB_API_URL={tunnel_url}")
        print("-"*70)
        print("🚀 Những ngày sau chỉ cần mở Colab bấm Chạy, web sẽ TỰ ĐỘNG KẾT NỐI mà không bao giờ cần sửa lại .env!\n")
    else:
        print("\n📋 HÃY SAO CHÉP 2 DÒNG DƯỚI ĐÂY DÁN VÀO FILE backend/.env TRÊN MÁY BẠN:")
        print("-"*70)
        print("USE_COLAB_GPU=true")
        print(f"COLAB_API_URL={tunnel_url}")
        print("-"*70 + "\n")
    print("💡 Giữ tab Google Colab này mở trong suốt quá trình sử dụng hệ thống.\n")
    try:
        if TUNNEL_METHOD == "ngrok" and NGROK_AUTHTOKEN.strip() and "ngrok" in tunnel_url:
            from pyngrok import ngrok
            ngrok.get_ngrok_process().proc.wait()
        else:
            tunnel_proc.wait()
    except KeyboardInterrupt:
        print("🛑 Đã dừng worker.")
else:
    print("⚠️ Không tìm thấy URL kết nối. Vui lòng thử khởi động lại ô này.")

🔍 [1/5] Đang kiểm tra card đồ hoạ GPU...


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'